# 1. Train — original CRM data (`data/crm/`)

Same steps as `python -m src.train`, one cell per step, using the functions in `src/train.py`.
Models are saved to `reports/crm/final/models/` for `2_evaluate.ipynb` and `3_predict.ipynb`. No test row is scored here.

Kernel: the project environment `.venv-crm` (Python 3.12).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from src import train
from src.data.crm import CRM_DIR


## Settings

In [ ]:
DATA_DIR = CRM_DIR   # real Maven CRM tables
QUICK = False            # True: tiny budgets, smoke test only (writes a *smoke* folder)
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

## 1. Load, validate and split the data

Joins the four CRM tables, builds the 18 pre-close features and splits closed deals by engagement date.
Training and validation keep only deals whose outcome was known before the next period.

In [ ]:
out, manifest, dataset, split = train.start(OUTPUT, quick=QUICK, data_dir=DATA_DIR)
report = dataset.report
pd.Series({
    "closed deals (labeled)": report["rows"],
    "won": report["positives"],
    "lost": report["negatives"],
    "open deals": report["open_rows"],
    "scorable Engaging deals": report["scorable_open_rows"],
    "scorable deals missing an account": report["scorable_missing_account"],
    "features": len(dataset.feature_columns),
})

In [ ]:
pd.DataFrame(manifest["split"], index=["value"]).T

## 2. Fit the five models

Each cell fits one model on the training rows, picks its class threshold on validation (macro-F1)
and shows its validation metrics.

In [ ]:
# Feature sets: "raw" (18 inputs) and, on the real CRM data, "history" (+ strictly prior outcomes
# from the frozen training archive). Every model is fitted on each feature set with the same split.
variants = train.feature_variants(dataset, split, train.default_feature_sets(DATA_DIR))
fitted = {name: {} for name in variants}
COLUMNS = ["model", "roc_auc", "accuracy", "precision", "recall", "f1", "brier", "threshold"]

def fit(model):
    for name, data in variants.items():
        fitted[name][model] = train.fit_one(model, data, split, quick=QUICK)
    return pd.DataFrame([{"feature_set": n, **fitted[n][model][2]} for n in variants])[["feature_set"] + COLUMNS].round(4)

print({name: len(d.feature_columns) for name, d in variants.items()})
fit("dummy_prior")

In [ ]:
fit("logistic_regression")

In [ ]:
fit("random_forest")

In [ ]:
fit("mlp")

In [ ]:
fit("tabnet")

In [ ]:
fit("catboost")

### Learning curves of the neural models

Early stopping keeps the epoch with the best validation ROC-AUC (dashed line).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, name in zip(axes, ["mlp", "tabnet"]):
    model = fitted["raw"][name][0]
    ax.plot(range(1, len(model.history["validation_auc"]) + 1), model.history["validation_auc"])
    ax.axvline(model.settings["best_epoch"], ls="--", color="gray")
    ax.set(title=name, xlabel="Epoch", ylabel="Validation ROC-AUC")
plt.tight_layout()

## 3. Compare on validation, select, calibrate and save

The model with the highest validation ROC-AUC is selected (ties: lower Brier, then name).
A sigmoid calibration is fitted on validation around the frozen selected model.

In [ ]:
pd.DataFrame([{"feature_set": n, **f[2]} for n, models in fitted.items() for f in models.values()])[["feature_set"] + COLUMNS].sort_values("roc_auc", ascending=False).round(4)

In [ ]:
trained = train.finish(out, manifest, variants, split, fitted)   # validation-only lock -> models/selection_lock.json
print("Selected:", trained.feature_set, "/", trained.selected)
print("Raw threshold:", trained.thresholds[trained.selected], "-> calibrated:", round(trained.decision_threshold, 4))
print("Calibration slope / intercept:", round(trained.calibrated.slope, 4), round(trained.calibrated.intercept, 4))
sorted(p.name for p in out.models.iterdir())

Next: `2_evaluate.ipynb` for test metrics, then `3_predict.ipynb` for scores.